In [1]:
import pandas as pd
import os

BASE_DIR = os.getcwd()
OUTPUT_CSV = "output_csv_transformer"

OUTPUT_PATH = os.path.join(BASE_DIR , OUTPUT_CSV)
print("OUTPUT_PATH :: ", OUTPUT_PATH)

coord_df = pd.read_csv('masking - cam_masking_coordinates_complete_status.csv', sep = ',')

coord_df['Prev_cam'] = coord_df['Prev_cam'].astype(int)
coord_df['Next_cam'] = coord_df['Next_cam'].astype(int)

coord_df[['Route','Cam_id','Prev_cam','Next_cam' ]] = coord_df[['Route','Cam_id','Prev_cam','Next_cam' ]].astype(str) #read the integers into strings
unique_route = coord_df['Route'].unique();
unique_route =['b_PIE', 'b_Woodlands']
print(unique_route)

OUTPUT_PATH ::  C:\Users\Padmapriya\Documents\ML_projects\counting_vehicles_4jul_2023\Counting-vehicles-by-direction\output_csv_transformer
['b_PIE', 'b_Woodlands']


In [2]:
MONTH_LIST=["3", "4", "5", "6"]
REGRESSION_INPUT_FILE ="regression_input_all_transformer.csv"

In [15]:
MONTH_LIST=["2"]
REGRESSION_INPUT_FILE ="regression_input_feb_transformer.csv"

In [15]:
MONTH_LIST=["7"]
REGRESSION_INPUT_FILE ="regression_input_jul_transformer.csv"

In [16]:
route=[]
cam_id_values=[]
timestamp=[]
speed =[]

total_car_count=[]
total_truck_count =[]
total_bus_count =[]

curr_Vehicles_no_filter=[]
prev_Vehicles_no_filter =[]
next_Vehicles_no_filter =[]

curr_Vehicles_acc_more_than_30 =[]
prev_Vehicles_acc_more_than_30 =[]
next_Vehicles_acc_more_than_30 =[]

curr_bbratio_no_filter=[]
prev_bbratio_no_filter=[]
next_bbratio_no_filter=[]

curr_bbratio_acc_more_than_30=[]
prev_bbratio_acc_more_than_30=[]
next_bbratio_acc_more_than_30=[]

first_camera=[]
last_camera=[]
day_list=[]
route_order = []
month_list=[]


for MONTH in MONTH_LIST:
    
    OUTPUT_CSV_PATH = os.path.join(OUTPUT_PATH , MONTH)
    print("OUTPUT_CSV_PATH :: ", OUTPUT_CSV_PATH)

    for ROUTE_DIR in unique_route:
        print(ROUTE_DIR)
        pdf  = coord_df.loc[(coord_df['Route']==ROUTE_DIR)]
        
        for CAMERA_ID in pdf['Cam_id']:
            df  = pdf.loc[(pdf['Route']==ROUTE_DIR) & (pdf['Cam_id']==CAMERA_ID)] 

            prev_ = df['Prev_cam'].values[0]
            next_ = df['Next_cam'].values[0]
            order =  df['route_order'].values[0]

            cam_file_prefix = 'combined_'+ROUTE_DIR+'_'+CAMERA_ID+'_'

            for filename in os.listdir(OUTPUT_CSV_PATH):
                if filename.startswith(cam_file_prefix):                
                    curr_df = pd.read_csv( os.path.join(OUTPUT_CSV_PATH, filename), sep = ',')
                    
                    route.extend(curr_df['Route'])
                    cam_id_values.extend(curr_df['Camera_id'])
                    timestamp.extend(curr_df['Timestamp_sg_time']) 
                    speed.extend(curr_df['avg_speed'])
                    
                    total_car_count.extend(curr_df['car'])
                    total_truck_count.extend(curr_df['truck'])
                    total_bus_count.extend(curr_df['bus'])
                    
                    day = pd.to_datetime(curr_df['Timestamp_sg_time'][0]).day_name()
                    day_list.extend([day] * curr_df.shape[0])

                    vehicles_val = curr_df['Vehicles_no_filer']
                    curr_Vehicles_no_filter.extend(vehicles_val)
                    
                    vehicles_val_more_than_30 = curr_df['Vehicles_acc_more_than_30']
                    curr_Vehicles_acc_more_than_30.extend(vehicles_val_more_than_30)
                    
                    
                    bbratio_val = curr_df['bbratio_no_filter']
                    curr_bbratio_no_filter.extend(bbratio_val)
                    print(bbratio_val)
                    
                    bbratio_val_more_than_30 = curr_df['bbratio_acc_more_than_30']
                    curr_bbratio_acc_more_than_30.extend(bbratio_val_more_than_30)

                    listofzeros = [0] * curr_df.shape[0];
                    listofones = [1] * curr_df.shape[0];
                    
                    route_order.extend( [order] * curr_df.shape[0])
                    month_list.extend([MONTH] * curr_df.shape[0]);
                    
                    if(prev_!='0'):            
                        prev_filename = filename.replace(CAMERA_ID, prev_)
                        prev_df = pd.read_csv( os.path.join(OUTPUT_CSV_PATH, prev_filename), sep = ',')
                        prev_Vehicles_no_filter.extend(prev_df['Vehicles_no_filer'])
                        prev_Vehicles_acc_more_than_30.extend(prev_df['Vehicles_acc_more_than_30'])
                        prev_bbratio_no_filter.extend(prev_df['bbratio_no_filter'])
                        prev_bbratio_acc_more_than_30.extend(prev_df['bbratio_acc_more_than_30'])

                        first_camera.extend(listofzeros)

                    else:
                        prev_Vehicles_no_filter.extend(vehicles_val)
                        prev_Vehicles_acc_more_than_30.extend(vehicles_val_more_than_30)
                        prev_bbratio_no_filter.extend(bbratio_val)
                        prev_bbratio_acc_more_than_30.extend(bbratio_val_more_than_30)

                        first_camera.extend(listofones)

                    if(next_!='0'):            
                        next_filename = filename.replace(CAMERA_ID, next_)
                        next_df = pd.read_csv( os.path.join(OUTPUT_CSV_PATH, next_filename), sep = ',')
                        next_Vehicles_no_filter.extend(next_df['Vehicles_no_filer'])
                        next_Vehicles_acc_more_than_30.extend(next_df['Vehicles_acc_more_than_30'])
                        next_bbratio_no_filter.extend(next_df['bbratio_no_filter'])
                        next_bbratio_acc_more_than_30.extend(next_df['bbratio_acc_more_than_30'])

                        last_camera.extend(listofzeros)

                    else:
                        next_Vehicles_no_filter.extend(vehicles_val)
                        next_Vehicles_acc_more_than_30.extend(vehicles_val_more_than_30)
                        next_bbratio_no_filter.extend(bbratio_val)
                        next_bbratio_acc_more_than_30.extend(bbratio_val_more_than_30)
                        
                        last_camera.extend(listofones)


OUTPUT_CSV_PATH ::  C:\Users\Padmapriya\Documents\ML_projects\counting_vehicles_4jul_2023\Counting-vehicles-by-direction\output_csv_transformer\2
b_PIE
0      1.504402
1      0.240498
2      2.834780
3      2.834780
4           NaN
         ...   
224    0.195553
225    2.799360
226    2.799360
227    0.805827
228    0.413926
Name: bbratio_no_filter, Length: 229, dtype: float64
0           NaN
1      7.666863
2      0.257662
3      0.103978
4      0.066177
         ...   
224    0.241589
225    0.241589
226    1.338712
227    1.393876
228    3.905908
Name: bbratio_no_filter, Length: 229, dtype: float64
0      0.086813
1           NaN
2           NaN
3           NaN
4      9.248057
         ...   
224    1.411239
225    0.307567
226    5.408326
227    5.408326
228    0.259547
Name: bbratio_no_filter, Length: 229, dtype: float64
0           NaN
1      3.189276
2      9.210355
3      9.210355
4           NaN
         ...   
224    0.254686
225    1.350717
226    0.618211
227         NaN
2

0      2.482968
1      8.909424
2      3.393610
3      3.393610
4      0.249809
         ...   
224    9.202464
225    2.064916
226    1.294619
227    1.294619
228         NaN
Name: bbratio_no_filter, Length: 229, dtype: float64
0      0.097751
1      0.598008
2      1.087829
3      1.087829
4      2.149251
         ...   
224    2.354337
225    0.727065
226    4.073397
227         NaN
228         NaN
Name: bbratio_no_filter, Length: 229, dtype: float64
0      0.588850
1      0.250235
2      4.351530
3      4.351530
4           NaN
         ...   
224    1.521428
225    1.521428
226    5.651473
227    5.651473
228    9.340466
Name: bbratio_no_filter, Length: 229, dtype: float64
0      0.584804
1      2.544302
2      0.277920
3      0.277920
4      0.544341
         ...   
224    7.438043
225    7.438043
226    3.741383
227    1.046088
228    0.493868
Name: bbratio_no_filter, Length: 229, dtype: float64
0           NaN
1           NaN
2           NaN
3           NaN
4           NaN
    

In [17]:
#https://stackoverflow.com/questions/30522724/take-multiple-lists-into-dataframe
output_df = pd.DataFrame(
    {
     'Route': route,
     'Camera_id': cam_id_values,
     'Timestamp': timestamp,
     'Y_Speed' : speed,
     'car' :  total_car_count,
     'truck' :  total_truck_count,
     'bus' : total_bus_count,

     'X1_Vehicles_no_filter': curr_Vehicles_no_filter,
     'X2_Vehicles_acc_more_than_30' : curr_Vehicles_acc_more_than_30,
     'X3_bbratio_no_filter' : curr_bbratio_no_filter,
     'X4_bbratio_acc_more_than_30' : curr_bbratio_acc_more_than_30,
     
     'X5_Day_name': day_list,
        
     'X6_Vehicles_no_filter_cam_idPlus1': next_Vehicles_no_filter,
     'X7_Vehicles_acc_more_than_30_cam_idPlus1': next_Vehicles_acc_more_than_30,
     'X8_bbratio_no_filter_cam_idPlus1': next_bbratio_no_filter,
     'X9_bbratio_acc_more_than_30_cam_idPlus1' : next_bbratio_acc_more_than_30,
     
     'X10_Vehicles_no_filter_cam_idMinus1': prev_Vehicles_no_filter,
     'X11_Vehicles_acc_more_than_30_cam_idMinus1' : prev_Vehicles_acc_more_than_30,
     'X12_bbratio_no_filter_cam_idMinus1' : prev_bbratio_no_filter,
     'X13_bbratio_acc_more_than_30_cam_idMinus1' : prev_bbratio_acc_more_than_30,

     'First_camera': first_camera,
     'Last_camera': last_camera,
     'route_order': route_order,
     'X14_Month': month_list
    })

In [18]:
output_df.describe()

,Camera_id,Y_Speed,car,truck,bus,X1_Vehicles_no_filter,X2_Vehicles_acc_more_than_30,X3_bbratio_no_filter,X4_bbratio_acc_more_than_30,X6_Vehicles_no_filter_cam_idPlus1,X7_Vehicles_acc_more_than_30_cam_idPlus1,X8_bbratio_no_filter_cam_idPlus1,X9_bbratio_acc_more_than_30_cam_idPlus1,X10_Vehicles_no_filter_cam_idMinus1,X11_Vehicles_acc_more_than_30_cam_idMinus1,X12_bbratio_no_filter_cam_idMinus1,X13_bbratio_acc_more_than_30_cam_idMinus1,First_camera,Last_camera,route_order
count,11221.000000,11221.0,10793.000000,10793.000000,10793.000000,10793.000000,10793.000000,10793.000000,10793.000000,10759.000000,10759.000000,10759.000000,10759.000000,10827.000000,10827.000000,10827.000000,10827.000000,11221.000000,11221.000000,11221.000000
mean,2706.000000,0.0,11.408135,2.660706,0.116001,14.184842,14.184842,9.911976,9.911976,14.447811,14.447811,9.712047,9.712047,13.923525,13.923525,10.110649,10.110649,0.285714,0.285714,2.285714
std,1.511925,0.0,7.073119,2.631943,0.396515,8.065155,8.065155,11.011992,11.011992,8.870086,8.870086,11.156491,11.156491,7.166843,7.166843,10.862853,10.862853,0.451774,0.451774,1.030203
min,2703.000000,0.0,0.000000,0.000000,0.000000,1.000000,1.000000,0.023613,0.023613,1.000000,1.000000,0.023613,0.023613,1.000000,1.000000,0.037760,0.037760,0.000000,0.000000,1.000000
25%,2705.000000,0.0,6.000000,1.000000,0.000000,8.000000,8.000000,2.729144,2.729144,7.000000,7.000000,2.424910,2.424910,9.000000,9.000000,3.061237,3.061237,0.000000,0.000000,1.000000
50%,2706.000000,0.0,11.000000,2.000000,0.000000,14.000000,14.000000,6.680068,6.680068,13.000000,13.000000,6.303603,6.303603,14.000000,14.000000,7.108752,7.108752,0.000000,0.000000,2.000000
75%,2707.000000,0.0,16.000000,4.000000,0.000000,20.000000,20.000000,13.273080,13.273080,21.000000,21.000000,12.909986,12.909986,19.000000,19.000000,13.686107,13.686107,1.000000,1.000000,3.000000
max,2708.000000,0.0,41.000000,22.000000,7.000000,43.000000,43.000000,146.284280,146.284280,43.000000,43.000000,146.284280,146.284280,41.000000,41.000000,146.284280,146.284280,1.000000,1.000000,4.000000


In [19]:
output_df.head(2)

,Route,Camera_id,Timestamp,Y_Speed,car,truck,bus,X1_Vehicles_no_filter,X2_Vehicles_acc_more_than_30,X3_bbratio_no_filter,...,X8_bbratio_no_filter_cam_idPlus1,X9_bbratio_acc_more_than_30_cam_idPlus1,X10_Vehicles_no_filter_cam_idMinus1,X11_Vehicles_acc_more_than_30_cam_idMinus1,X12_bbratio_no_filter_cam_idMinus1,X13_bbratio_acc_more_than_30_cam_idMinus1,First_camera,Last_camera,route_order,X14_Month
0,b_PIE,2705,2022-02-14 04:50:00,0,3.0,2.0,0.0,5.0,5.0,1.504402,...,1.504402,1.504402,3.0,3.0,0.48200,0.48200,0,1,3,2
1,b_PIE,2705,2022-02-14 04:55:00,0,5.0,0.0,0.0,5.0,5.0,0.240498,...,0.240498,0.240498,7.0,7.0,3.21954,3.21954,0,1,3,2


In [20]:
output_df.to_csv(REGRESSION_INPUT_FILE, index=False)

In [21]:
# Filter the DataFrame
def calculate_total_counts(df_pivoted, vehicle_type):
    # Extract columns with the given vehicle_type prefix
    columns_to_sum = [col for col in df_pivoted.columns if col.startswith(vehicle_type)]

    # Sum the columns for each timestamp
    total_column_name = f'total_{vehicle_type}'
    df_pivoted[total_column_name] = df_pivoted[columns_to_sum].sum(axis=1)

    return df_pivoted



# Pivot the DataFrame
def count_total_vehicles_per_route(output_df, route):
    
    temp_df = output_df[output_df['Route'] == route]
    
    df_pivoted = temp_df.pivot(index='Timestamp', columns='Camera_id', values=['car', 'truck', 'bus'])
    
    # Flatten the MultiIndex columns
    df_pivoted.columns = [f'{col[0]}_{col[1]}' for col in df_pivoted.columns]

    # Reset the index to make Timestamp a regular column
    df_pivoted.reset_index(inplace=True)

    # If there are missing values, fill them with 0
    df_pivoted.fillna(0, inplace=True)

    # Display the resulting DataFrame
    print(df_pivoted.head(3))

    # Use the function for 'car'
    df_pivoted = calculate_total_counts(df_pivoted, 'car')

    # Use the function for 'truck'
    df_pivoted = calculate_total_counts(df_pivoted, 'truck')

    # Use the function for 'bus'
    df_pivoted = calculate_total_counts(df_pivoted, 'bus')

    df_pivoted['Route'] = route
    
    return df_pivoted
    
    
df_pivoted = count_total_vehicles_per_route(output_df, 'b_PIE')
df_pivoted

             Timestamp  car_2705  car_2706  car_2707  truck_2705  truck_2706  \
0  2022-02-14 04:50:00       3.0       2.0       3.0         2.0         1.0   
1  2022-02-14 04:55:00       5.0       3.0       6.0         0.0         0.0   
2  2022-02-14 05:00:00       3.0       2.0       7.0         0.0         0.0   

   truck_2707  bus_2705  bus_2706  bus_2707  
0         0.0       0.0       0.0       0.0  
1         1.0       0.0       0.0       0.0  
2         0.0       0.0       0.0       0.0  


,Timestamp,car_2705,car_2706,car_2707,truck_2705,truck_2706,truck_2707,bus_2705,bus_2706,bus_2707,total_car,total_truck,total_bus,Route
0,2022-02-14 04:50:00,3.0,2.0,3.0,2.0,1.0,0.0,0.0,0.0,0.0,8.0,3.0,0.0,b_PIE
1,2022-02-14 04:55:00,5.0,3.0,6.0,0.0,0.0,1.0,0.0,0.0,0.0,14.0,1.0,0.0,b_PIE
2,2022-02-14 05:00:00,3.0,2.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0,0.0,0.0,b_PIE
3,2022-02-14 05:05:00,3.0,2.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0,0.0,0.0,b_PIE
4,2022-02-14 05:10:00,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,b_PIE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1598,2022-02-20 23:30:00,14.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,1.0,18.0,0.0,1.0,b_PIE
1599,2022-02-20 23:35:00,14.0,3.0,4.0,0.0,0.0,0.0,0.0,0.0,1.0,21.0,0.0,1.0,b_PIE
1600,2022-02-20 23:40:00,3.0,1.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,5.0,2.0,0.0,b_PIE
1601,2022-02-20 23:45:00,1.0,8.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,b_PIE


In [22]:
df_pivoted.columns

Index(['Timestamp', 'car_2705', 'car_2706', 'car_2707', 'truck_2705',
       'truck_2706', 'truck_2707', 'bus_2705', 'bus_2706', 'bus_2707',
       'total_car', 'total_truck', 'total_bus', 'Route'],
      dtype='object')

In [23]:
df_pivoted.describe()

,car_2705,car_2706,car_2707,truck_2705,truck_2706,truck_2707,bus_2705,bus_2706,bus_2707,total_car,total_truck,total_bus
count,1603.000000,1603.000000,1603.000000,1603.000000,1603.000000,1603.000000,1603.000000,1603.000000,1603.000000,1603.000000,1603.000000,1603.000000
mean,18.812227,9.224579,12.708047,2.398628,2.988771,3.266999,0.087336,0.112289,0.265128,40.744853,8.654398,0.464754
std,8.439492,5.872296,5.567953,2.343514,3.082490,2.884775,0.334989,0.407321,0.574272,15.395385,6.515349,0.796669
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,14.000000,4.000000,9.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,32.000000,3.000000,0.000000
50%,20.000000,9.000000,13.000000,2.000000,2.000000,3.000000,0.000000,0.000000,0.000000,43.000000,8.000000,0.000000
75%,25.000000,14.000000,16.000000,4.000000,5.000000,5.000000,0.000000,0.000000,0.000000,52.000000,13.000000,1.000000
max,41.000000,30.000000,29.000000,17.000000,22.000000,18.000000,4.000000,4.000000,5.000000,78.000000,37.000000,5.000000


In [24]:
# Filter the DataFrame
def calculate_total_counts(df_pivoted, vehicle_type):
    # Extract columns with the given vehicle_type prefix
    columns_to_sum = [col for col in df_pivoted.columns if col.startswith(vehicle_type)]

    # Sum the columns for each timestamp
    total_column_name = f'total_{vehicle_type}'
    df_pivoted[total_column_name] = df_pivoted[columns_to_sum].sum(axis=1)

    return df_pivoted

# Pivot the DataFrame
def count_total_vehicles_per_route(output_df, route):
    ## WE need data 
    temp_df = output_df[output_df['X5_Day_name'].isin(['Saturday', 'Sunday'])]
    temp_df = temp_df[temp_df['Route'] == route]
    
    df_pivoted = temp_df.pivot(index='Timestamp', columns='Camera_id', values=['car', 'truck', 'bus'])
    
    # Flatten the MultiIndex columns
    df_pivoted.columns = [f'{col[0]}_{col[1]}' for col in df_pivoted.columns]

    # Reset the index to make Timestamp a regular column
    df_pivoted.reset_index(inplace=True)

    # If there are missing values, fill them with 0
    df_pivoted.fillna(0, inplace=True)

    # Use the function for 'car'
    df_pivoted = calculate_total_counts(df_pivoted, 'car')

    # Use the function for 'truck'
    df_pivoted = calculate_total_counts(df_pivoted, 'truck')

    # Use the function for 'bus'
    df_pivoted = calculate_total_counts(df_pivoted, 'bus')

    df_pivoted['Route'] = route
    
    # Extract hour and minute from 'Timestamp' and create 'Time' column
    df_pivoted['Time'] = pd.to_datetime(df_pivoted['Timestamp']).dt.strftime('%H:%M')

    # Group by 'Time' and calculate the mean for each unique time
    df_average = df_pivoted.groupby('Time').mean().reset_index()
    
    print(df_average.columns)
    # Round off the values to the next integer
    
    if(route=='b_Woodlands'):
        columns_to_round =  ['car_2703', 'car_2706', 'car_2707', 'car_2708', 'truck_2703',
                             'truck_2706', 'truck_2707', 'truck_2708', 'bus_2703', 'bus_2706',
                             'bus_2707', 'bus_2708', 'total_car', 'total_truck', 'total_bus' ]
    else:
        columns_to_round = ['car_2705', 'car_2706', 'car_2707', 'truck_2705', 'truck_2706', 'truck_2707',
                        'bus_2705', 'bus_2706', 'bus_2707', 'total_car', 'total_truck', 'total_bus']

    df_average[columns_to_round] = np.ceil(df_average[columns_to_round]).astype(np.int16)

    df_average.to_csv(route+'Average_vehicles_per_Time_on_weekends_Transformer.csv',index=False)
    
    return df_average

# Calculate total counts and average values per route
df_average = count_total_vehicles_per_route(output_df, 'b_PIE')


# Display the resulting DataFrame
print(df_average.head(3)['Time'])

df_average

Index(['Time', 'car_2705', 'car_2706', 'car_2707', 'truck_2705', 'truck_2706',
       'truck_2707', 'bus_2705', 'bus_2706', 'bus_2707', 'total_car',
       'total_truck', 'total_bus'],
      dtype='object')
0    04:50
1    04:55
2    05:00
Name: Time, dtype: object


C:\Users\Padmapriya\AppData\Local\Temp\ipykernel_307060\2086066448.py:44: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df_average = df_pivoted.groupby('Time').mean().reset_index()


,Time,car_2705,car_2706,car_2707,truck_2705,truck_2706,truck_2707,bus_2705,bus_2706,bus_2707,total_car,total_truck,total_bus
0,04:50,1,2,2,0,0,0,0,0,0,4,0,0
1,04:55,8,1,5,1,0,1,0,0,0,14,1,0
2,05:00,2,0,2,2,0,0,0,0,0,3,2,0
3,05:05,2,0,2,2,0,0,0,0,0,3,2,0
4,05:10,3,1,2,0,0,1,0,0,0,6,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
224,23:30,11,1,3,0,0,0,0,0,1,15,0,1
225,23:35,11,3,3,0,0,0,0,0,1,17,0,1
226,23:40,8,2,5,1,0,0,0,0,0,15,1,0
227,23:45,7,5,5,0,0,0,0,0,0,17,0,0
